# TRM - Deep Supervision Final Experiment

This notebook implements the **Deep Supervision** training workflow for the Tiny Recursive Model (TRM).

### Key Features
- **N_sup Iterative Refinement**: The model "thinking" process is split into `N_sup` steps.
- **Iterative Weight Updates**: Gradients are backpropagated and optimizer steps are taken **inside** the loop, allowing the model to learn to improve its own latent state.
- **Truncated Gradient Flow**: Hidden states (`y`, `z`) are detached between supervision steps to prevent exploding gradients.

### Setup
Ensure you have the `TRM/` directory structure uploaded to Colab.

In [ ]:
# 1. Setup Environment
import sys
import os
import torch

# Ensure we can import from src
if 'google.colab' in sys.modules:
    # Assuming repo is cloned/uploaded to /content/TRM
    sys.path.append('/content/TRM')
    os.chdir('/content/TRM')
    print("Running on Colab, path updated.")

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# 2. Import Modules
from src.model import TRMModel, create_trm_model
from src.training.trainer import TRMTrainer
from src.data.maze import MazeDataset
from src.data.sudoku import SudokuDataset
from torch.utils.data import DataLoader

In [ ]:
# 3. Configure Experiment

# TASK: 'maze' or 'sudoku'
TASK = 'maze'

if TASK == 'maze':
    # Maze Config
    config = {
        'dim': 256,
        'n_layers': 2,
        'n_heads': 4,
        'n_latent': 6,     # Inner latent steps
        'T_cycles': 2,     # Recursion depth per supervision step
        'vocab_size': 5,   # Maze vocabulary
        'max_seq_len': 100,
        'N_sup': 16,       # Deep Supervision Steps (The big loop)
        'batch_size': 128,
        'lr': 1e-4,
        'max_iters': 10000
    }
    # Dummy dataset for quick test if no files present
    # In real usage, use actual dataset
    print("Initializing Maze Task...")
    # dataset = MazeDataset(size=10, num_samples=100000) 
    
else:
    # Sudoku Config
    config = {
        'dim': 512,
        'n_layers': 2,
        'n_heads': 8,
        'n_latent': 6,
        'T_cycles': 3,
        'vocab_size': 10,
        'max_seq_len': 81,
        'N_sup': 16,
        'batch_size': 64,
        'lr': 1e-4,
        'max_iters': 50000
    }
    print("Initializing Sudoku Task...")

In [ ]:
# 4. Initialize Model and Trainer

model = create_trm_model(
    dim=config['dim'],
    n_layers=config['n_layers'],
    n_heads=config['n_heads'],
    n_latent=config['n_latent'],
    T_cycles=config['T_cycles'],
    vocab_size=config['vocab_size'],
    max_seq_len=config['max_seq_len']
)

print(f"\nCreated TRM Model with Deep Supervision N_sup={config['N_sup']}")

In [ ]:
# 5. Create Dummy Dataloader (For Verification)
# Replace this with actual dataset loading logic

x_dummy = torch.randint(0, config['vocab_size'], (1000, config['max_seq_len']))
y_true = torch.randint(0, config['vocab_size'], (1000, config['max_seq_len']))
y_init = torch.zeros_like(y_true)

from torch.utils.data import TensorDataset

dataset = TensorDataset(x_dummy, y_init, y_true)

def collate_fn(batch):
    xs, yis, yts = zip(*batch)
    return {
        "x_tokens": torch.stack(xs),
        "y_init_tokens": torch.stack(yis),
        "y_true": torch.stack(yts)
    }

dataloader = DataLoader(dataset, batch_size=config['batch_size'], shuffle=True, collate_fn=collate_fn)

print("Dataloader ready.")

In [ ]:
# 6. Start Training

trainer = TRMTrainer(
    model=model,
    train_loader=dataloader,
    lr=config['lr'],
    N_sup=config['N_sup'],
    device=device,
    use_amp=(device == 'cuda')
)

trainer.train(max_iters=100) # Run short test

In [ ]:
# 7. Verify Results
print("Training check complete. If loss decreased, Deep Supervision is working!")